# 0505 · 00 · Preparar el entorno

Laboratorio 0505 · Ingesta de ficheros JSON.

Ejecuta este notebook con **Run all** antes de empezar. Hace cuatro cosas:

1. Comprueba que en la carpeta `datasets/events-kafka/` del repositorio están los ficheros de eventos con sus líneas esperadas.
2. Crea el catálogo `lab0505` con dos schemas:
   - `bronze`: donde se crean las tablas del laboratorio.
   - `upload`: con el volumen `upload`, la carpeta de aterrizaje de los ficheros.
3. Deja el laboratorio en su **estado inicial**: borra las tablas de una ejecución anterior, vacía el volumen y copia:
   - `000.json` y `001.json` a `events-kafka/`: los **2.252 eventos** de la demo;
   - `006.json` y `007.json` a `events-kafka-extra/`: solo para la extensión opcional del final.
4. Verifica que `events-kafka/` contiene 2.252 eventos.

Si quieres repetir el laboratorio desde cero, vuelve a ejecutarlo: siempre deja el mismo punto de partida.

In [ ]:
import os

CATALOGO = "lab0505"
VOLUMEN_UPLOAD = f"/Volumes/{CATALOGO}/upload/upload"

# La carpeta de este notebook es la del laboratorio dentro de tu Git folder;
# los datasets viven un nivel más arriba, en la raíz del repositorio.
RUTA_LAB = os.getcwd()
if not RUTA_LAB.startswith("/Workspace"):
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    RUTA_LAB = "/Workspace" + os.path.dirname(ctx.notebookPath().get())
RUTA_DATASETS = f"{os.path.dirname(RUTA_LAB)}/datasets/events-kafka"

print("Laboratorio:", RUTA_LAB)
print("Datasets:   ", RUTA_DATASETS)
print("Catálogo:   ", CATALOGO)

## 1 · Comprobar los ficheros del repositorio

Cada fichero es JSON Lines: un evento por línea. Se cuentan las líneas y se comprueba que cada una es un JSON válido con los seis campos del sobre de Kafka.

In [ ]:
import json

# carpeta del volumen -> {fichero: eventos esperados}
FICHEROS = {
    "events-kafka":       {"000.json": 1126, "001.json": 1126},
    "events-kafka-extra": {"006.json": 500, "007.json": 600},
}
CAMPOS = {"key", "offset", "partition", "timestamp", "topic", "value"}

if not os.path.isdir(RUTA_DATASETS):
    raise FileNotFoundError(
        f"No encuentro {RUTA_DATASETS}. Comprueba que tu Git folder está actualizado "
        "y que la carpeta datasets/ está en la raíz del repositorio.")

errores = []
for carpeta, ficheros in FICHEROS.items():
    for fichero, esperados in ficheros.items():
        ruta = f"{RUTA_DATASETS}/{fichero}"
        if not os.path.exists(ruta):
            errores.append(f"falta {fichero}")
            continue
        with open(ruta, encoding="utf-8") as f:
            lineas = [l for l in f.read().splitlines() if l.strip()]
        malos = sum(1 for l in lineas if set(json.loads(l)) != CAMPOS)
        print(f"{fichero}: {len(lineas)} eventos  {'ok' if len(lineas) == esperados and not malos else 'REVISAR'}")
        if len(lineas) != esperados or malos:
            errores.append(f"{fichero}: {len(lineas)} eventos (esperaba {esperados}), {malos} sin el sobre de Kafka")

if errores:
    raise ValueError("Ficheros incorrectos: " + "; ".join(errores))

## 2 · Catálogo, schemas y volumen

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
for schema in ["bronze", "upload"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.upload.upload")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOGO}"))

## 3 · Volver al estado inicial

In [ ]:
for tabla in ["kafka_events_bronze_raw", "kafka_events_bronze_decoded",
              "kafka_events_bronze_string_flattened", "kafka_events_bronze_struct",
              "bronze_explode_array", "kafka_events_bronze_variant"]:
    spark.sql(f"DROP TABLE IF EXISTS {CATALOGO}.bronze.{tabla}")

for item in dbutils.fs.ls(VOLUMEN_UPLOAD):
    dbutils.fs.rm(item.path, recurse=True)
print("vaciado", VOLUMEN_UPLOAD)

## 4 · Cargar los ficheros y verificar

In [ ]:
import shutil

for carpeta, ficheros in FICHEROS.items():
    os.makedirs(f"{VOLUMEN_UPLOAD}/{carpeta}", exist_ok=True)
    for fichero in ficheros:
        shutil.copyfile(f"{RUTA_DATASETS}/{fichero}", f"{VOLUMEN_UPLOAD}/{carpeta}/{fichero}")
        print("copiado", f"{carpeta}/{fichero}")

eventos = spark.sql(f"""
    SELECT count(*) AS eventos
    FROM read_files('{VOLUMEN_UPLOAD}/events-kafka', format => 'json')
""").first()["eventos"]

assert eventos == 2252, f"events-kafka tiene {eventos} eventos, esperaba 2252"
print(f"Entorno listo: {VOLUMEN_UPLOAD}/events-kafka contiene {eventos} eventos.")